In [21]:
import logging
import joblib
import mlflow
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [22]:
MODEL_NAME = "heart_disease_pred_model"
MODEL_STAGE = "None"  # since you're not using staging yet

LOCAL_MODEL_PATH = Path("../models/heart_disease_pred_model.pkl")

In [23]:
def load_model_from_mlflow():
    try:
        logger.info("Trying MLflow Registry...")

        dagshub_uri = os.getenv("MLFLOW_TRACKING_URI")
        print(f"MLFLOW_TRACKING_URI from .env: {dagshub_uri}")


        if dagshub_uri:
            mlflow.set_tracking_uri(dagshub_uri)
            print(f"Using DagsHub MLflow URI: {dagshub_uri}")
        else:
            tracking_path = Path("../mlruns").resolve()
            mlflow.set_tracking_uri(f"file:///{tracking_path.as_posix()}")
            print(f"Using Local MLflow URI: {tracking_path}")

        model_uri = f"models:/{MODEL_NAME}/{MODEL_STAGE}"
        model = mlflow.sklearn.load_model(model_uri)

        logger.info("Loaded from MLflow")
        return model

    except Exception as e:
        logger.warning(f"MLflow load failed: {e}")
        return None

In [24]:
def load_model_from_local():
    try:
        logger.info("Trying local model...")

        if not LOCAL_MODEL_PATH.exists():
            raise FileNotFoundError(f"{LOCAL_MODEL_PATH} not found")

        model = joblib.load(LOCAL_MODEL_PATH)

        logger.info("Loaded from local file")
        return model

    except Exception as e:
        logger.error(f"Local load failed: {e}")
        raise

In [25]:
def load_model():
    model = load_model_from_mlflow()

    if model is not None:
        return model

    logger.info("Falling back to local...")
    return load_model_from_local()

In [26]:
model = load_model()

print("Model type:", type(model))

2026-05-17 20:37:09,047 - INFO - Trying MLflow Registry...


MLFLOW_TRACKING_URI from .env: https://dagshub.com/Bhooshaan-2025cs05032/DMML-Assignment.mlflow
Using DagsHub MLflow URI: https://dagshub.com/Bhooshaan-2025cs05032/DMML-Assignment.mlflow


2026-05-17 20:37:14,471 - INFO - Loaded from MLflow


Model type: <class 'sklearn.pipeline.Pipeline'>


In [27]:
def predict_heart_disease(data: dict) -> dict:
    try:
        # Convert to DataFrame
        df = pd.DataFrame([data])

        # Prediction
        pred = model.predict(df)[0]

        # Confidence
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(df)[0]
            confidence = probs[pred]
        else:
            confidence = None

        result = {
            "prediction": int(pred),
            "confidence": float(round(confidence, 4)) if confidence is not None else None
        }

        logger.info(f"Prediction result: {result}")

        return result

    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        raise

In [28]:
healthy_sample = {
    "age": 40,
    "sex": 0,
    "cp": 1,           # less severe chest pain
    "trestbps": 110,
    "chol": 180,
    "fbs": 0,
    "restecg": 0,
    "thalach": 170,    # high max heart rate (good sign)
    "exang": 0,
    "oldpeak": 0.0,    # no ST depression
    "slope": 2,
    "ca": 0,
    "thal": 2
}

disease_sample = {
    "age": 65,
    "sex": 1,
    "cp": 4,           # severe chest pain
    "trestbps": 160,
    "chol": 300,
    "fbs": 1,
    "restecg": 2,
    "thalach": 100,    # low max heart rate
    "exang": 1,
    "oldpeak": 4.0,    # strong ST depression
    "slope": 0,
    "ca": 3,
    "thal": 3
}


print("Healthy Sample Prediction:")
prediction = predict_heart_disease(healthy_sample).get("prediction")
confidence = predict_heart_disease(healthy_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})


print("Disease Sample Prediction:")
prediction = predict_heart_disease(disease_sample).get("prediction")
confidence = predict_heart_disease(disease_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})


2026-05-17 20:37:22,172 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.8302}


2026-05-17 20:37:22,202 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.8302}
2026-05-17 20:37:22,234 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.872}
2026-05-17 20:37:22,268 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.872}


Healthy Sample Prediction:
{'prediction': 0, 'confidence': 0.8302}
Disease Sample Prediction:
{'prediction': 1, 'confidence': 0.872}


# Reproducability

In [29]:
# Load a specific model run and return it for prediction

def load_specific_model_and_predict(run_id: str):
    try:
        dagshub_uri = os.getenv("MLFLOW_TRACKING_URI")
        print(f"MLFLOW_TRACKING_URI from .env: {dagshub_uri}")


        if dagshub_uri:
            mlflow.set_tracking_uri(dagshub_uri)
            print(f"Using DagsHub MLflow URI: {dagshub_uri}")
        else:
            tracking_path = Path("../mlruns").resolve()
            mlflow.set_tracking_uri(f"file:///{tracking_path.as_posix()}")
            print(f"Using Local MLflow URI: {tracking_path}")
            
        logger.info(f"Loading model from run_id: {run_id}")
        model_uri = f"runs:/{run_id}/model"
        model = mlflow.sklearn.load_model(model_uri)
        logger.info("Model loaded successfully")

        # Print Details of Loaded Model
        print(f"Loaded model from run_id: {run_id}")
        print(f"Model type: {type(model)}")
        print(f"Model details: {model}")

        return model
    except Exception as e:
        logger.error(f"Failed to load model from run_id {run_id}: {e}")
        raise

In [30]:
model = load_specific_model_and_predict("d28ec7a15b684558a8bcf356562afeef")


2026-05-17 20:37:27,660 - INFO - Loading model from run_id: d28ec7a15b684558a8bcf356562afeef


MLFLOW_TRACKING_URI from .env: https://dagshub.com/Bhooshaan-2025cs05032/DMML-Assignment.mlflow
Using DagsHub MLflow URI: https://dagshub.com/Bhooshaan-2025cs05032/DMML-Assignment.mlflow


2026-05-17 20:37:31,202 - INFO - Model loaded successfully


Loaded model from run_id: d28ec7a15b684558a8bcf356562afeef
Model type: <class 'sklearn.pipeline.Pipeline'>
Model details: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'sex', 'trestbps',
                                                   'chol', 'fbs', 'thalach',
                                                   'exang', 'oldpeak', 'ca']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cp', 'restecg', 'slope',
                                                   'thal'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, random_state=42))])


In [31]:
print("Healthy Sample Prediction:")
prediction = predict_heart_disease(healthy_sample).get("prediction")
confidence = predict_heart_disease(healthy_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})


print("Disease Sample Prediction:")
prediction = predict_heart_disease(disease_sample).get("prediction")
confidence = predict_heart_disease(disease_sample).get("confidence")
print({
    "prediction": int(prediction),
    "confidence": float(round(confidence, 4))
})

2026-05-17 20:37:32,918 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.85}


2026-05-17 20:37:32,945 - INFO - Prediction result: {'prediction': 0, 'confidence': 0.85}
2026-05-17 20:37:32,965 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.86}
2026-05-17 20:37:32,987 - INFO - Prediction result: {'prediction': 1, 'confidence': 0.86}


Healthy Sample Prediction:
{'prediction': 0, 'confidence': 0.85}
Disease Sample Prediction:
{'prediction': 1, 'confidence': 0.86}
